# Tarefa 1 - Laboratorio 1: Wikipedia

Notebook organizado no estilo das aulas da professora.

Link escolhido para a atividade:

`https://en.wikipedia.org/wiki/Vancouver_Stock_Exchange`


## Instalacao no Google Colab

Esta celula instala a biblioteca necessaria para executar o notebook no Colab.


In [ ]:
!pip install beautifulsoup4


## Objetivo

1. Baixar o HTML da pagina inicial.
2. Baixar os HTMLs de todos os links internos `/wiki/` encontrados.
3. Extrair titulo, primeiro paragrafo, links internos e links de imagens.
4. Salvar os resultados em dois arquivos CSV com timestamp.


In [ ]:
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError
from urllib.parse import unquote
from bs4 import BeautifulSoup
from datetime import datetime
import csv
import os
import time


## Configuracao inicial

Aqui definimos a URL principal, a URL base da Wikipedia e os nomes dos arquivos de saida.


In [ ]:
URL_INICIAL = "https://en.wikipedia.org/wiki/Vancouver_Stock_Exchange"
BASE_URL = "https://en.wikipedia.org"
PASTA_HTML = "htmls_lab1_vancouver"
CSV_PAGINAS = "paginas_wikipedia.csv"
CSV_IMAGENS = "imagens_wikipedia.csv"
HEADERS = {"User-Agent": "aula-cpa"}

print("Link inicial:", URL_INICIAL)


## Funcao para abrir a pagina com tratamento de erros

Esta parte segue diretamente o exemplo da professora com `urllib.request` e `urllib.error`.


In [ ]:
def get_pagina(url):
    try:
        req = Request(url, headers=HEADERS)
        html = urlopen(req)
    except HTTPError as erro:
        print("Houve um erro na obtencao da pagina!", erro)
        return None
    except URLError as erro:
        print("Ocorreu um erro no servidor!", erro)
        return None
    else:
        print("Consegui abrir a pagina:", url)
        return html.read()


## Funcoes auxiliares

Estas funcoes ajudam a salvar os HTMLs, montar nomes de arquivo e extrair os dados pedidos no enunciado.


In [ ]:
def corrigir_nome_arquivo(nome):
    nome = unquote(nome)
    nome = nome.replace("/", "_")
    nome = nome.replace("\\", "_")
    nome = nome.replace(":", "_")
    nome = nome.replace("*", "_")
    nome = nome.replace("?", "_")
    nome = nome.replace('"', "_")
    nome = nome.replace("<", "_")
    nome = nome.replace(">", "_")
    nome = nome.replace("|", "_")
    return nome


def salvar_html(url, conteudo, pasta=PASTA_HTML):
    os.makedirs(pasta, exist_ok=True)
    nome_arquivo = url.split("/")[-1]
    nome_arquivo = corrigir_nome_arquivo(nome_arquivo) + ".html"
    caminho = os.path.join(pasta, nome_arquivo)

    with open(caminho, "w", encoding="utf-8") as arquivo:
        arquivo.write(conteudo.decode("utf-8", errors="ignore"))

    print("HTML salvo com sucesso:", caminho)
    return caminho


def extrair_links_internos(soup):
    links_internos = []

    for link in soup.select("a[href^='/wiki/']"):
        href = link.get("href")

        if href is None:
            continue

        if "#" in href:
            href = href.split("#")[0]

        if "?" in href:
            href = href.split("?")[0]

        if href == "/wiki/":
            continue

        link_completo = BASE_URL + href

        if link_completo not in links_internos:
            links_internos.append(link_completo)

    return links_internos


def extrair_links_imagens(soup):
    links_imagens = []

    for img in soup.select("img"):
        if img.has_attr("src"):
            src = img["src"]

            if src.startswith("//"):
                src = "https:" + src
            elif src.startswith("/"):
                src = BASE_URL + src

            if src not in links_imagens:
                links_imagens.append(src)

    return links_imagens


def extrair_primeiro_paragrafo(soup):
    paragrafos = soup.find_all("p")

    for paragrafo in paragrafos:
        texto = paragrafo.get_text(" ", strip=True)
        if texto != "":
            return texto

    return ""


def extrair_titulo(soup):
    if soup.head and soup.head.title and soup.head.title.string:
        return soup.head.title.string.strip()

    return "Sem titulo"


## Baixar a pagina inicial e coletar os links internos

Esta celula baixa o HTML inicial, salva o arquivo e identifica todos os links internos `/wiki/`.


In [ ]:
pagina_inicial = get_pagina(URL_INICIAL)

if pagina_inicial is not None:
    soup_inicial = BeautifulSoup(pagina_inicial, "html.parser")
    timestamp_inicial = datetime.now().isoformat()
    caminho_inicial = salvar_html(URL_INICIAL, pagina_inicial)
    links_internos_iniciais = extrair_links_internos(soup_inicial)

    print("Titulo da pagina inicial:", extrair_titulo(soup_inicial))
    print("Primeiro paragrafo da pagina inicial:")
    print(extrair_primeiro_paragrafo(soup_inicial))
    print("Quantidade de links internos encontrados:", len(links_internos_iniciais))
    print("Exemplo de 5 links internos:")
    for link in links_internos_iniciais[:5]:
        print(link)


## Baixar os HTMLs dos links internos

Cada pagina baixada tambem recebe um timestamp, como solicitado no enunciado.


In [ ]:
paginas_baixadas = []

if pagina_inicial is not None:
    paginas_baixadas.append(
        {
            "url": URL_INICIAL,
            "arquivo_html": caminho_inicial,
            "timestamp": timestamp_inicial,
        }
    )

    for indice, link in enumerate(links_internos_iniciais, start=1):
        if link == URL_INICIAL:
            continue

        print(f"[{indice}/{len(links_internos_iniciais)}] Baixando: {link}")
        pagina = get_pagina(link)

        if pagina is not None:
            timestamp = datetime.now().isoformat()
            caminho = salvar_html(link, pagina)
            paginas_baixadas.append(
                {
                    "url": link,
                    "arquivo_html": caminho,
                    "timestamp": timestamp,
                }
            )

        time.sleep(0.5)

print("Total de paginas baixadas:", len(paginas_baixadas))


## Extrair os dados pedidos do enunciado

Agora lemos os HTMLs salvos e montamos duas tabelas:
- uma tabela com os dados completos de cada pagina;
- uma tabela apenas com os links das imagens.


In [ ]:
dados_paginas = []
dados_imagens = []

for pagina in paginas_baixadas:
    with open(pagina["arquivo_html"], "r", encoding="utf-8") as arquivo_html:
        soup = BeautifulSoup(arquivo_html, "html.parser")

    titulo = extrair_titulo(soup)
    primeiro_paragrafo = extrair_primeiro_paragrafo(soup)
    links_internos = extrair_links_internos(soup)
    links_imagens = extrair_links_imagens(soup)

    print("Pagina analisada:", pagina["url"])
    print("Titulo:", titulo)
    print("Quantidade de links internos:", len(links_internos))
    print("Quantidade de imagens:", len(links_imagens))
    print("-" * 60)

    dados_paginas.append(
        [
            pagina["url"],
            os.path.basename(pagina["arquivo_html"]),
            titulo,
            primeiro_paragrafo,
            " | ".join(links_internos),
            " | ".join(links_imagens),
            pagina["timestamp"],
        ]
    )

    for imagem in links_imagens:
        dados_imagens.append(
            [
                pagina["url"],
                titulo,
                imagem,
                pagina["timestamp"],
            ]
        )

print("Total de linhas para o CSV de paginas:", len(dados_paginas))
print("Total de linhas para o CSV de imagens:", len(dados_imagens))


## Salvar os arquivos CSV

O primeiro CSV guarda os dados principais de cada pagina.
O segundo CSV guarda apenas os links das imagens.


In [ ]:
with open(CSV_PAGINAS, "w", newline="", encoding="utf-8") as arquivo_paginas:
    escritor_paginas = csv.writer(arquivo_paginas)
    escritor_paginas.writerow(
        [
            "page_url",
            "html_file",
            "title",
            "first_paragraph",
            "internal_links",
            "image_links",
            "timestamp",
        ]
    )
    escritor_paginas.writerows(dados_paginas)

with open(CSV_IMAGENS, "w", newline="", encoding="utf-8") as arquivo_imagens:
    escritor_imagens = csv.writer(arquivo_imagens)
    escritor_imagens.writerow(["page_url", "page_title", "image_url", "timestamp"])
    escritor_imagens.writerows(dados_imagens)

print("CSV de paginas salvo em:", CSV_PAGINAS)
print("CSV de imagens salvo em:", CSV_IMAGENS)


## Conclusao

O notebook realiza todas as etapas pedidas na Tarefa 1:
- baixa a pagina inicial da Wikipedia;
- baixa os links internos encontrados;
- extrai titulo, primeiro paragrafo, imagens e links internos;
- salva os dados em CSV com timestamp.
